# Bolt reference-input validation audit

Offline ANL voltage/SOC checks and completed EDU map provenance. **CRUISE M was not executed.** The discharge battery model has not passed independent validation.

Data: native CSV channels plus TDMS Phase 1 indices; NREL Figure 12; EPA EDU completed map and original SwRI points. SOC channel identity, single-voltage EDU transfer and charge/discharge differences remain material assumptions.

In [1]:
from pathlib import Path
import json, subprocess, sys, hashlib
import pandas as pd
ROOT=Path.cwd()
SOURCE=ROOT.parent / "validation_sources/public_inputs/Bolt_CRUISE_M_DataKit"
RAW=ROOT.parent / "validation_sources/raw_reference/raw_tdms"
assert SOURCE.is_dir() and RAW.is_dir(), "Materialize the separate source archives and update SOURCE/RAW."
run=subprocess.run([sys.executable,str(ROOT/"scripts/audit_inputs.py"),"--source",str(SOURCE),"--raw",str(RAW)],capture_output=True,text=True,check=True)
(ROOT/"results/audit_run.json").write_text(run.stdout)
print("Raw-source numerical audit completed. No simulation execution.")
quality=json.loads((ROOT/"results/battery_data_quality.json").read_text())

Raw-source numerical audit completed. No simulation execution.


## Data quality and time alignment

Use the original Phase 1 row boundaries; deduplicate channel clocks by median; hold CAN observations; exclude invalid voltage/current rows and gaps >2 s from integration. Missing coverage is reported rather than silently filled. Numeric integration and declared phase totals are distinct.

In [2]:
pd.DataFrame([{k:r[k] for k in ["test","native_phase_rows","unique_meter_rows","invalid_voltage_or_current_rows","coverage_fraction","integrated_valid_meter_Wh","declared_phase_Wh"]} for r in quality])

,test,native_phase_rows,unique_meter_rows,invalid_voltage_or_current_rows,coverage_fraction,integrated_valid_meter_Wh,declared_phase_Wh
0,61905054,11450,11450,0,1.000000,2155.018046,2172.0
1,61905064,101251,101057,0,1.000000,46772.514710,46903.0
2,61905065,63956,63940,29709,0.535346,-60809.403754,-60319.0


## Battery reference equation and SOC meaning

The specific Figure 12 coefficient coordinate is **percent SOC**, despite the general report equation describing fractional SOC. Fractional input z requires coefficient conversion. Displayed and internal ECU SOC are separate observed channels; neither is established as NREL's exact coordinate or chemical SOC. The 5–95% priority window is a research choice, not a validated domain.

In [3]:
r=pd.read_csv(ROOT/"results/battery_voltage_by_SOC.csv")
r[(r.SOC_lower_inclusive_percent==5)&(r.SOC_upper_percent==95)][["test","SOC_channel","rows","RMSE_V","mean_bias_predicted_minus_measured_V","p95_abs_error_V"]]

,test,SOC_channel,rows,RMSE_V,mean_bias_predicted_minus_measured_V,p95_abs_error_V
3,61905054,internal_SOC_HPCM2_percent,7072,5.794104,5.772982,6.617400
6,61905054,internal_SOC_HPCM_percent,8080,5.318979,5.291080,6.219676
14,61905064,displayed_SOC_percent,95611,7.413374,7.282048,9.134600
21,61905064,internal_SOC_HPCM2_percent,99827,10.025014,9.266399,11.390093
28,61905064,internal_SOC_HPCM_percent,99806,9.917828,9.140902,11.311169
37,61905065,displayed_SOC_percent,30452,1.839176,0.299654,3.129773


In [4]:
pd.read_csv(ROOT/"results/display_to_internal_SOC_observed_fit.csv")

,test,internal_channel,display_low,display_high,unique_display_steps,slope,intercept_percent,mapping_RMSE_percentage_points,status
0,61905064,internal_SOC_HPCM2_percent,5.0973,79.9884,292,0.939738,4.643163,0.404036,EMPIRICAL_CHANNEL_CONVERSION_IN_THIS_TEST_ONLY_NOT_CHEMICAL_SOC
1,61905064,internal_SOC_HPCM_percent,5.0973,79.9884,292,0.935836,4.626949,0.391298,EMPIRICAL_CHANNEL_CONVERSION_IN_THIS_TEST_ONLY_NOT_CHEMICAL_SOC


## Completed-map provenance

Inside the original measurement convex hull does not mean a direct measured grid node. Regen loss uses EPA drive-loss mirror factor1.05, with no original regen observations. Reconstruction error against original fitting points is not independent validation. Runtime lookup uses bilinear loss, with Full load/power/axis constraints and no external extrapolation.

In [5]:
print(json.dumps(json.loads((ROOT/"results/bolt_map_audit.json").read_text()),indent=2))
pd.read_csv(ROOT/"results/bolt_map_support_counts.csv")

{
  "grid_nodes": 204,
  "runtime_allowed_grid_nodes": 151,
  "measured_source_points": 252,
  "reconstruction_RMSE_kW": 0.09360393020519912,
  "reconstruction_is_independent_validation": false,
  "regen_measured_source_points": 0,
  "source_regen_loss_mirror_factor": 1.05,
  "nominal_voltage_V": 400,
  "voltage_temperature_dependence_identified": false,
  "CRUISE_M_executed": false
}


,support_class,runtime_allowed,grid_nodes
0,COMPLETED_MAP_INSIDE_MEASURED_HULL,0,1
1,COMPLETED_MAP_INSIDE_MEASURED_HULL,1,63
2,COMPLETED_MAP_OUTSIDE_MEASURED_HULL,0,26
3,COMPLETED_MAP_OUTSIDE_MEASURED_HULL,1,18
4,REGEN_ESTIMATE_DRIVE_LOSS_MIRROR_1p05,0,26
5,REGEN_ESTIMATE_DRIVE_LOSS_MIRROR_1p05,1,70


## Evaluator checks and comparison gates

Synthetic fixtures verify dimensional integrals and refusal of invalid input or unmatched terminal battery energy. They do not establish vehicle-model accuracy. Independent battery/cycle results and actual CRUISE traces are still required. Fuel/genset bounds, transient fuel and depletion mission definitions are required for EREV fuel/range claims.

In [6]:
run=subprocess.run([sys.executable,str(ROOT/"scripts/check_evaluator.py")],capture_output=True,text=True,check=True)
json.loads((ROOT/"results/evaluator_checks.json").read_text())

{'test_origin': 'SYNTHETIC_FIXTURE_NOT_CRUISE_M', 'checks_passed': ['distance_units', 'battery_DC_energy_units', 'fuel_mass_units', 'unverified_metadata_rejected', 'outside_full_load_detected', 'duplicate_time_rejected', 'SOC_percent_rejected', 'unequal_terminal_energy_rejected'], 'CRUISE_M_executed': False}

## Takeaways

Charge reference replay is much closer than discharge replay; changing SOC channel alone does not fix discharge residuals. No full-SOC discharge acceptance domain has been established. Preserve equation/data lineage, identify discharge dynamics and coordinate/capacity on suitable data, validate on independent conditions, then compare like-for-like cycle results. Official wall-AC label values and battery-DC results need matching boundaries and procedures. EREV total range requires explicit initial inventories, stopping criteria and feasible speed tracking.